# LeetCode 3632 — Subarrays with XOR at Least K bằng Python

- Link: https://leetcode.com/problems/subarrays-with-xor-at-least-k/
- Độ khó: **Hard**
- Chủ đề: **XOR tiền tố**, **trie nhị phân** (0-1 trie) đếm cặp có XOR `≥ k`, đếm theo từng bit bằng bảng băm

> **Ghi chú:** Đây là bài **Premium**. Đề bài lấy từ bản doocs (leetcode.doocs.org); trang doocs không có mã nguồn, nên tên hàm `countXorSubarrays` là **suy luận** — cần đối chiếu skeleton thật nếu có tài khoản Premium.

Skeleton (suy luận):

```python
class Solution:
    def countXorSubarrays(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. XOR của mảng con `[i, j)` = `P[j] ⊕ P[i]` với `P` là XOR tiền tố ⇒ đếm **cặp tiền tố** `(i < j)` có `P[i] ⊕ P[j] ≥ k`.
2. Dùng **trie nhị phân** chứa các tiền tố đã gặp, mỗi nút lưu số lượng; truy vấn "bao nhiêu `p` có `p ⊕ x ≥ k`" bằng cách đi theo bit của `k` từ cao xuống.
3. Hiểu quy tắc: tại bit `b`, nếu `k_b = 0` thì mọi `p` làm bit `b` của `p ⊕ x` bằng 1 đều **chắc chắn lớn hơn** ⇒ cộng cả nhánh; rồi đi tiếp nhánh làm bit bằng 0.
4. Cách thay thế không cần trie: với mỗi bit `b` mà `k_b = 0`, đếm cặp có `(p ⊕ x) >> b == (k >> b) | 1` bằng `dict`.
5. Kiểm chứng bằng vét cạn `O(n²)`.

## 2. Đề bài & ví dụ

Cho `nums` (số không âm) và `k ≥ 0`. Đếm số mảng con liên tiếp có XOR mọi phần tử `≥ k`.

**Ví dụ 1:** `nums = [3,1,2,3], k = 2` → `6`.

**Ví dụ 2:** `nums = [0,0,0], k = 0` → `6`.

**Ràng buộc:** `n ≤ 10^5`, `nums[i], k ≤ 10^9` (30 bit).

In [1]:
from typing import List
from collections import defaultdict
import random, time

examples = [([3, 1, 2, 3], 2, 6), ([0, 0, 0], 0, 6)]
for a, k, exp in examples:
    print(a, k, "-> kỳ vọng", exp)

[3, 1, 2, 3] 2 -> kỳ vọng 6
[0, 0, 0] 0 -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. XOR tiền tố

`P[0] = 0`, `P[j] = nums[0] ⊕ … ⊕ nums[j−1]`. Vì `x ⊕ x = 0`: `nums[i] ⊕ … ⊕ nums[j−1] = P[j] ⊕ P[i]`.

In [2]:
nums = [3, 1, 2, 3]; P = [0]
for x in nums: P.append(P[-1] ^ x)
print("P =", P, "| XOR [1, 3) =", 1 ^ 2, "=", P[3] ^ P[1])

P = [0, 3, 2, 0, 3] | XOR [1, 3) = 3 = 3


### 3.2. So sánh `x ⊕ y` với `k` theo bit

So sánh hai số bắt đầu từ bit cao nhất. Gọi `z = x ⊕ y`. Duyệt `b` từ 29 xuống 0, giả sử các bit cao hơn của `z` **bằng** của `k`:

- `k_b = 0`: nếu `z_b = 1` thì `z > k` ngay (cộng hết); nếu `z_b = 0`, tiếp tục so bit thấp hơn.
- `k_b = 1`: phải có `z_b = 1` để còn hi vọng; `z_b = 0` thì `z < k` ⇒ loại.

Đi hết 30 bit mà vẫn bằng ⇒ `z = k` ⇒ thoả `≥`.

### 3.3. Trie nhị phân

Mỗi số là đường đi 30 bit từ gốc; mỗi nút lưu `cnt` = số số đi qua. Với `x` và bit `b`, nhánh làm `z_b = 1` là nhánh có bit `x_b ⊕ 1`; nhánh làm `z_b = 0` là nhánh có bit `x_b`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Duyệt `j = 1..n`: trước khi chèn `P[j]`, trie chứa `P[0..j−1]`; truy vấn số `p` trong trie có `p ⊕ P[j] ≥ k` — đó là số mảng con hợp lệ kết thúc tại `j − 1`. Cộng dồn.

Ví dụ 1: `P = [0, 3, 2, 0, 3]`, `k = 2`. Các cặp `(i<j)` có `P[i] ⊕ P[j] ≥ 2`: `(0,1):3, (0,2):2, (0,4):3, (1,3):3, (2,3):2, (3,4):3` ⇒ `6`. ✔

## 5. Cách 1 — Brute Force

In [3]:
def brute(nums, k):
    n = len(nums); c = 0
    for i in range(n):
        x = 0
        for j in range(i, n):
            x ^= nums[j]
            if x >= k: c += 1
    return c

for a, k, exp in examples:
    print(brute(a, k), "kỳ vọng", exp)

6 kỳ vọng 6
6 kỳ vọng 6


**Nút thắt:** `O(n²) = 10^{10}`.

## 6. Ý tưởng thuật toán / bước đột phá

Truy vấn trie cho `x` (`P[j]` hiện tại):

```
node = gốc; res = 0
for b in 29..0:
    xb = bit b của x; kb = bit b của k
    if kb == 0:
        res += cnt[con[node][xb ^ 1]]        # z_b = 1 > k_b: cả nhánh thoả
        node = con[node][xb]                 # z_b = 0: so tiếp
    else:
        node = con[node][xb ^ 1]             # bắt buộc z_b = 1
    if node không tồn tại: return res
return res + cnt[node]                       # z == k
```

Mỗi truy vấn/chèn `O(30)` ⇒ tổng `O(30 n)`.

## 7. Các cách tiếp theo

- **Cách 2:** trie nhị phân dùng mảng phẳng (lời giải nộp).
- **Cách 3:** đếm theo bit bằng `dict`: với mỗi bit `b` có `k_b = 0`, cặp có `(p ⊕ x) >> b == (k >> b) | 1` là cặp "lớn hơn `k` lần đầu tại bit `b`"; cộng thêm cặp có `p ⊕ x == k`.
- **Cách 4:** phần bù — đếm cặp có XOR `< k` bằng trie rồi lấy `C(n+1, 2)` trừ đi.

### 7.1. Cách 2 — Trie mảng phẳng (lời giải chính)

In [4]:
def trie_count(nums, k, B=30):
    n = len(nums)
    size = (n + 1) * B + 2
    ch0 = [0] * size; ch1 = [0] * size; cnt = [0] * size; nodes = 1     # nút 0 = gốc

    def insert(x):
        nonlocal nodes
        u = 0; cnt[0] += 1
        for b in range(B - 1, -1, -1):
            if x >> b & 1:
                if not ch1[u]: ch1[u] = nodes; nodes += 1
                u = ch1[u]
            else:
                if not ch0[u]: ch0[u] = nodes; nodes += 1
                u = ch0[u]
            cnt[u] += 1

    def query(x):
        u = 0; res = 0
        for b in range(B - 1, -1, -1):
            xb = x >> b & 1
            same = ch1[u] if xb else ch0[u]; other = ch0[u] if xb else ch1[u]
            if k >> b & 1:
                u = other
            else:
                if other: res += cnt[other]
                u = same
            if not u: return res
        return res + cnt[u]

    insert(0); total = 0; x = 0
    for v in nums:
        x ^= v; total += query(x); insert(x)
    return total

for a, k, exp in examples:
    print(trie_count(a, k), "kỳ vọng", exp)

6 kỳ vọng 6
6 kỳ vọng 6


### 7.2. Cách 3 — Đếm theo bit bằng `dict`

In [5]:
def per_bit_hash(nums, k, B=30):
    P = [0]
    for v in nums: P.append(P[-1] ^ v)
    total = 0
    # cặp có z == k
    seen = defaultdict(int)
    for x in P: total += seen[x ^ k]; seen[x] += 1
    # cặp có z > k, lần đầu vượt tại bit b (k_b = 0)
    for b in range(B):
        if k >> b & 1: continue
        T = (k >> b) | 1
        seen = defaultdict(int)
        for x in P:
            total += seen[(x >> b) ^ T]; seen[x >> b] += 1
    return total

for a, k, exp in examples:
    print(per_bit_hash(a, k), "kỳ vọng", exp)

6 kỳ vọng 6
6 kỳ vọng 6


**Vì sao đúng?** `z > k` ⇔ tồn tại **duy nhất** bit `b` (bit khác nhau cao nhất) sao cho `z` và `k` bằng nhau trên các bit `> b`, `z_b = 1`, `k_b = 0` ⇔ `z >> b == (k >> b) | 1`. Các điều kiện theo `b` khác nhau loại trừ nhau ⇒ cộng thẳng.

### 7.3. Cách 4 — Phần bù: đếm `< k`

In [6]:
def complement(nums, k, B=30):
    class Node:
        __slots__ = ("c", "n")
        def __init__(self): self.c = [None, None]; self.n = 0
    root = Node()
    def insert(x):
        u = root; u.n += 1
        for b in range(B - 1, -1, -1):
            t = x >> b & 1
            if u.c[t] is None: u.c[t] = Node()
            u = u.c[t]; u.n += 1
    def less(x):                                   # số p có p ^ x < k
        u = root; res = 0
        for b in range(B - 1, -1, -1):
            if u is None: return res
            xb = x >> b & 1
            if k >> b & 1:
                if u.c[xb] is not None: res += u.c[xb].n          # z_b = 0 < 1: cả nhánh nhỏ hơn
                u = u.c[xb ^ 1]
            else:
                u = u.c[xb]
        return res
    insert(0); x = 0; small = 0
    for v in nums:
        x ^= v; small += less(x); insert(x)
    n = len(nums)
    return n * (n + 1) // 2 - small

for a, k, exp in examples:
    print(complement(a, k), "kỳ vọng", exp)

6 kỳ vọng 6
6 kỳ vọng 6


## 8. TRACE từng bước

In [7]:
def trace(nums, k, B=3):
    P = [0]
    for v in nums: P.append(P[-1] ^ v)
    print("P =", P, "| k =", format(k, f"0{B}b"))
    stored = [0]
    for j in range(1, len(P)):
        x = P[j]; ok = [p for p in stored if p ^ x >= k]
        print(f"j={j}: x = {x:0{B}b}; tiền tố đã có {[format(p, f'0{B}b') for p in stored]} -> thoả {[format(p, f'0{B}b') for p in ok]} (+{len(ok)})")
        stored.append(x)
    print("tổng:", brute(nums, k))

trace([3, 1, 2, 3], 2)

P = [0, 3, 2, 0, 3] | k = 010
j=1: x = 011; tiền tố đã có ['000'] -> thoả ['000'] (+1)
j=2: x = 010; tiền tố đã có ['000', '011'] -> thoả ['000'] (+1)
j=3: x = 000; tiền tố đã có ['000', '011', '010'] -> thoả ['011', '010'] (+2)
j=4: x = 011; tiền tố đã có ['000', '011', '010', '000'] -> thoả ['000', '000'] (+2)
tổng: 6


In [8]:
# Đóng góp theo từng bit ở Cách 3
P = [0]
for v in [5, 1, 6, 2, 7]: P.append(P[-1] ^ v)
k = 4
for b in range(3):
    if k >> b & 1: continue
    T = (k >> b) | 1
    pairs = [(i, j) for i in range(len(P)) for j in range(i + 1, len(P)) if (P[i] ^ P[j]) >> b == T]
    print(f"bit {b}: z >> {b} == {T:b} -> {len(pairs)} cặp")
print("cặp z == k:", sum(1 for i in range(len(P)) for j in range(i + 1, len(P)) if P[i] ^ P[j] == k), "| tổng:", per_bit_hash([5, 1, 6, 2, 7], 4), "| brute:", brute([5, 1, 6, 2, 7], 4))

bit 0: z >> 0 == 101 -> 3 cặp
bit 1: z >> 1 == 11 -> 4 cặp
cặp z == k: 2 | tổng: 9 | brute: 9


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def countXorSubarrays(self, nums: List[int], k: int) -> int:
        B = 30
        size = (len(nums) + 1) * B + 2
        ch0 = [0] * size
        ch1 = [0] * size
        cnt = [0] * size
        nodes = 1
        total = 0
        x = 0
        for step in range(len(nums) + 1):
            if step:
                x ^= nums[step - 1]
                u = 0; res = 0
                for b in range(B - 1, -1, -1):
                    if x >> b & 1:
                        same, other = ch1[u], ch0[u]
                    else:
                        same, other = ch0[u], ch1[u]
                    if k >> b & 1:
                        u = other
                    else:
                        if other:
                            res += cnt[other]
                        u = same
                    if not u:
                        break
                else:
                    res += cnt[u]
                total += res
            u = 0
            cnt[0] += 1
            for b in range(B - 1, -1, -1):
                if x >> b & 1:
                    if not ch1[u]:
                        ch1[u] = nodes; nodes += 1
                    u = ch1[u]
                else:
                    if not ch0[u]:
                        ch0[u] = nodes; nodes += 1
                    u = ch0[u]
                cnt[u] += 1
        return total

sol = Solution()
for a, k, exp in examples:
    got = sol.countXorSubarrays(a, k)
    print(a, k, "->", got, "✔" if got == exp else "✘")

[3, 1, 2, 3] 2 -> 6 ✔
[0, 0, 0] 0 -> 6 ✔


## 10. Vì sao thuật toán đúng

1. **Song ánh:** mảng con `[i, j)` ↔ cặp `i < j` của dãy tiền tố; XOR mảng con = `P[i] ⊕ P[j]`.
2. **Truy vấn trie:** bất biến — trước bước `b`, nút `u` đại diện các `p` đã lưu sao cho `p ⊕ x` **bằng** `k` trên mọi bit `> b`. Tại bit `b`: nếu `k_b = 0`, nhánh `other` cho `z_b = 1 > 0` ⇒ mọi `p` trong đó thoả `z > k` ⇒ cộng `cnt[other]`; nhánh `same` giữ bất biến. Nếu `k_b = 1`, chỉ nhánh `other` (`z_b = 1`) giữ bất biến; nhánh `same` cho `z < k` ⇒ bỏ. Hết bit: các `p` còn lại có `z = k` ⇒ cộng.
3. **Mỗi `p` được đếm đúng một lần** (tại bit khác biệt đầu tiên, hoặc ở cuối nếu bằng).
4. Chèn `P[j]` **sau** khi truy vấn ⇒ chỉ đếm `i < j`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n²)` | `O(1)` |
| 2. Trie mảng | `O(30 n)` | `O(30 n)` |
| 3. Đếm theo bit (dict) | `O(30 n)` (hằng số dict) | `O(n)` |
| 4. Phần bù (trie đối tượng) | `O(30 n)` | `O(30 n)` |

## 12. Bộ test

In [10]:
METHODS = [brute, trie_count, per_bit_hash, complement, lambda a, k: Solution().countXorSubarrays(a, k)]
for a, k, exp in examples:
    assert all(f(a, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().countXorSubarrays([7], 8) == 0
assert Solution().countXorSubarrays([7], 7) == 1
assert Solution().countXorSubarrays([1, 2, 3], 0) == 6                    # k = 0: mọi mảng con
assert Solution().countXorSubarrays([2**29, 2**29], 2**29) == 2
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên
rng = random.Random(3632)
for _ in range(1500):
    a = [rng.randint(0, 15) for _ in range(rng.randint(1, 15))]; k = rng.randint(0, 16)
    b = brute(a, k)
    assert all(f(a, k) == b for f in METHODS[1:]), (a, k)
for _ in range(200):
    a = [rng.randint(0, 10**9) for _ in range(rng.randint(1, 40))]; k = rng.randint(0, 10**9)
    assert trie_count(a, k) == brute(a, k) == per_bit_hash(a, k)
print("1700 test ngẫu nhiên: OK")

1700 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn
rng = random.Random(1)
a = [rng.randint(0, 10**9) for _ in range(10**5)]; k = rng.randint(0, 10**9)
t0 = time.time(); x = Solution().countXorSubarrays(a, k); t1 = time.time()
y = per_bit_hash(a, k); t2 = time.time()
print(f"đáp án {x} | trie {t1 - t0:.2f}s | đếm theo bit {t2 - t1:.2f}s | khớp: {x == y}")

đáp án 3101175012 | trie 1.02s | đếm theo bit 0.43s | khớp: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên tiền tố rỗng `P[0] = 0`

In [14]:
def wrong_no_p0(nums, k):
    P = []; x = 0
    for v in nums: x ^= v; P.append(x)                                   # SAI: thiếu 0
    return sum(1 for i in range(len(P)) for j in range(i + 1, len(P)) if P[i] ^ P[j] >= k)

print("sai:", wrong_no_p0([3, 1, 2, 3], 2), "| đúng:", brute([3, 1, 2, 3], 2))

sai: 3 | đúng: 6


### Lỗi 2 — Không cộng các `p` cho `z == k` ở cuối (đếm `> k` thay vì `≥ k`)

In [15]:
def wrong_strict(nums, k):
    return trie_count(nums, k + 1)                                       # tương đương đếm > k

print("sai:", wrong_strict([3, 1, 2, 3], 2), "| đúng:", brute([3, 1, 2, 3], 2))

sai: 4 | đúng: 6


### Lỗi 3 — Khi `k_b = 1` lại cộng cả nhánh

In [16]:
def wrong_branch(nums, k, B=30):
    P = [0]
    for v in nums: P.append(P[-1] ^ v)
    total = 0
    for j in range(1, len(P)):
        for i in range(j):
            z = P[i] ^ P[j]; ok = None
            for b in range(B - 1, -1, -1):
                zb, kb = z >> b & 1, k >> b & 1
                if zb != kb: ok = True; break                            # SAI: z_b = 0 < k_b = 1 cũng bị coi là "lớn hơn"
            total += 1 if ok in (True, None) else 0
    return total

print("sai:", wrong_branch([3, 1, 2, 3], 2), "| đúng:", brute([3, 1, 2, 3], 2))

sai: 10 | đúng: 6


### Lỗi 4 — Dùng `nums[i] ⊕ nums[j]` (cặp phần tử) thay cho XOR mảng con

In [17]:
def wrong_pairs(nums, k):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n) if (nums[i] ^ nums[j] if i != j else nums[i]) >= k)   # SAI

rng = random.Random(23)
while True:
    a = [rng.randint(0, 7) for _ in range(rng.randint(3, 6))]; kk = rng.randint(0, 8)
    if wrong_pairs(a, kk) != brute(a, kk): break
print("phản ví dụ:", a, kk, "| sai:", wrong_pairs(a, kk), "| đúng:", brute(a, kk))

phản ví dụ: [2, 3, 4, 7, 0] 3 | sai: 11 | đúng: 8


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "XOR mảng con" ⇒ XOR tiền tố ⇒ bài **cặp**.
- "Số cặp có `x ⊕ y` so với `k`" ⇒ **0-1 trie** đi theo bit của `k` (hoặc đếm theo bit bằng bảng băm).
- `≥`, `>`, `<`, `≤` chỉ khác ở việc có cộng `cnt` của nút cuối hay không — kiểm tra kỹ.
- Python: trie bằng **mảng phẳng** nhanh và ít bộ nhớ hơn đối tượng.

## 15. Biến thể của bài

1. **XOR lớn nhất của mảng con** (LC 421-style trên tiền tố): đi nhánh ngược bit.
2. **Số cặp có XOR trong `[low, high]`** (LC 1803): `f(high + 1) − f(low)` với `f(t)` = số cặp `< t`.
3. **Mảng con có XOR đúng bằng `k`**: bảng băm `cnt[x ^ k]`.
4. **Truy vấn online "bao nhiêu mảng con kết thúc tại `j` có XOR ≥ k"**: chính là `query(P[j])`.

In [18]:
# Biến thể 2: số cặp phần tử (i < j) có low <= nums[i]^nums[j] <= high (LC 1803)
def count_pairs_range(nums, low, high):
    def less_than(t):
        root = [None, None, 0]; res = 0
        for x in nums:
            u = root
            for b in range(15, -1, -1):
                if u is None: break
                xb = x >> b & 1
                if t >> b & 1:
                    if u[xb] is not None: res += u[xb][2]
                    u = u[xb ^ 1]
                else: u = u[xb]
            u = root; u[2] += 1
            for b in range(15, -1, -1):
                xb = x >> b & 1
                if u[xb] is None: u[xb] = [None, None, 0]
                u = u[xb]; u[2] += 1
        return res
    return less_than(high + 1) - less_than(low)

rng = random.Random(15)
for _ in range(300):
    a = [rng.randint(0, 50) for _ in range(rng.randint(1, 12))]; lo = rng.randint(0, 40); hi = rng.randint(lo, 64)
    assert count_pairs_range(a, lo, hi) == sum(1 for i in range(len(a)) for j in range(i + 1, len(a)) if lo <= a[i] ^ a[j] <= hi)
print("Biến thể 2 OK:", count_pairs_range([1, 4, 2, 7], 2, 6))

Biến thể 2 OK: 6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi mảng con | `O(n²)` | Kiểm chứng |
| 2. Trie mảng | Đi theo bit `k` | `O(30n)` | **Lời giải chuẩn** |
| 3. Đếm theo bit | Bit khác biệt đầu tiên | `O(30n)` | Không muốn viết trie |
| 4. Phần bù | Đếm `< k` | `O(30n)` | Khi bài hỏi `≤`/`<` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countXorSubarrays([1,1,1], 1)`.

**Bài 2 (tính tay).** Liệt kê `P` và đếm cho `nums = [4,4,4], k = 4`.

**Bài 3 (điền TODO).** Viết `count_xor_equal(nums, k)` — số mảng con có XOR **đúng bằng** `k`.

**Bài 4 (tự viết lại).** Viết `count_xor_less(nums, k)` — số mảng con có XOR `< k` — bằng trie mảng phẳng.

**Bài 5 (mini challenge).** Tìm XOR lớn nhất của một mảng con.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_xor_equal(nums, k):
    # TODO
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def max_subarray_xor(nums):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `P = [0,1,0,1]`. (2) Cặp có XOR `≥ 1` là cặp khác nhau: `(0,1),(0,3),(1,2),(2,3)`. (3) `4`.

**Bài 2.** (1) `P = [0,4,0,4]`. (2) XOR các cặp: `4,0,4,4,0,4`. (3) `4`.

**Bài 3.** (1) Bảng băm đếm tiền tố. (2) `total += cnt[x ^ k]` trước khi thêm `x`. (3) Kiểm chứng vét cạn.

**Bài 4.** (1) `C(n+1, 2) − countXorSubarrays`. (2) Hoặc viết trực tiếp như `complement.less`. (3) So sánh hai cách.

**Bài 5.** (1) Trie tiền tố. (2) Với mỗi `x`, đi nhánh ngược bit nếu có. (3) Lấy max.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().countXorSubarrays([1, 1, 1], 1) == 4 == brute([1, 1, 1], 1)
assert Solution().countXorSubarrays([4, 4, 4], 4) == 4 == brute([4, 4, 4], 4)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def count_xor_equal_sol(nums, k):
    cnt = defaultdict(int); cnt[0] = 1; x = 0; total = 0
    for v in nums: x ^= v; total += cnt[x ^ k]; cnt[x] += 1
    return total

def count_xor_less_sol(nums, k):
    n = len(nums); return n * (n + 1) // 2 - Solution().countXorSubarrays(nums, k)

rng = random.Random(4)
for _ in range(500):
    a = [rng.randint(0, 15) for _ in range(rng.randint(1, 12))]; k = rng.randint(0, 16)
    n = len(a); subs = []
    for i in range(n):
        x = 0
        for j in range(i, n): x ^= a[j]; subs.append(x)
    assert count_xor_equal_sol(a, k) == subs.count(k)
    assert count_xor_less_sol(a, k) == sum(1 for s in subs if s < k)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5
def max_subarray_xor_sol(nums, B=30):
    root = [None, None]; best = 0; x = 0
    def insert(v):
        u = root
        for b in range(B - 1, -1, -1):
            t = v >> b & 1
            if u[t] is None: u[t] = [None, None]
            u = u[t]
    insert(0)
    for v in nums:
        x ^= v; u = root; cur = 0
        for b in range(B - 1, -1, -1):
            t = x >> b & 1
            if u[t ^ 1] is not None: cur |= 1 << b; u = u[t ^ 1]
            else: u = u[t]
        best = max(best, cur); insert(x)
    return best

rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(0, 63) for _ in range(rng.randint(1, 12))]
    exp = 0
    for i in range(len(a)):
        x = 0
        for j in range(i, len(a)): x ^= a[j]; exp = max(exp, x)
    assert max_subarray_xor_sol(a) == exp
print("Bài 5 OK:", max_subarray_xor_sol([3, 1, 2, 3]))

Bài 5 OK: 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
insert(0); x = 0; total = 0
for v in nums:
    x ^= v
    # query: số p trong trie có p ^ x >= k
    u = gốc; res = 0
    for b in 29..0:
        same, other = con theo bit x_b, con theo bit 1 - x_b
        if k_b == 1: u = other
        else: res += cnt[other]; u = same
        if u rỗng: break
    else: res += cnt[u]          # z == k
    total += res; insert(x)
```

**Checklist:**

- [ ] Có tiền tố rỗng `0`?
- [ ] `≥ k`: cộng `cnt` nút cuối khi đi hết bit?
- [ ] `k_b = 1` thì **không** cộng nhánh nào?
- [ ] Truy vấn **trước** khi chèn?
- [ ] Đủ 30 bit?